In [1]:
# Homework 3 Duncan Newman
# STAT 604

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gammaln
from scipy.optimize import minimize
from mpl_toolkits.mplot3d import Axes3D

data = np.array([87,119,112,111,114,105,116,124,130,101,115,119,88,116,112,106,102,120,95,109,104,96,111,112,87,115,127,111,123,113,134,92,81,90,92,124,123,121,106,120,133,110,103,137,93,115,98,121,111,108,91,98,110,118,105,104,113,104,119,125,126,115,108,127,102,98,105,110,124,116,105,115,115,98,118,115,153,108,116,101,113,98,120,93,127,116,103,118,118,105,101,93,99,90,106,87,120,116,122,92])



In [2]:
# 1a. Negative Log-Likelihood

def neg_log_likelihood(param, x):
    alpha, beta = param
    if alpha <= 0 or beta <= 0:
        return np.inf
    n = len(x)
    return n*(alpha*np.log(beta)+gammaln(alpha))-(alpha-1)*np.sum(np.log(x))+np.sum(x)/beta
    

In [3]:
#1d. Method of Moments (use these values for part b)

mean_x = np.mean(data)
var_x = np.var(data, ddof=1)

alpha_mom = mean_x**2 / var_x
beta_mom = var_x/mean_x

print("Method of Moments Estimations:")
print(f"alpha_MOM = {alpha_mom:.6f}")
print(f"beta_MOM = {beta_mom:.6f}")


Method of Moments Estimations:
alpha_MOM = 74.888490
beta_MOM = 1.470319


In [4]:
#1c. MLE using scipy.optimize

# Method 1: L-BFGS-B
res_lbfgs = minimize(
    neg_log_likelihood,
    x0=[alpha_mom, beta_mom],
    args=(data,),
    method='L-BFGS-B',
    bounds=[(1e-6, None), (1e-6, None)]
)

alpha_mle_lbfgs, beta_mle_lbfgs = res_lbfgs.x

print("\nMLE using L-BFGS-B:")
print(f"alpha_MLE = {alpha_mle_lbfgs:.6f}")
print(f"beta_MLE  = {beta_mle_lbfgs:.6f}")

# Method 2: Nelder-Mead
res_nm = minimize(
    neg_log_likelihood,
    x0=[alpha_mom, beta_mom],
    args=(data,),
    method='Nelder-Mead'
)

alpha_mle_nm, beta_mle_nm = res_nm.x

print("\nMLE using Nelder-Mead:")
print(f"alpha_MLE = {alpha_mle_nm:.6f}")
print(f"beta_MLE  = {beta_mle_nm:.6f}")


MLE using L-BFGS-B:
alpha_MLE = 74.888503
beta_MLE  = 1.470316

MLE using Nelder-Mead:
alpha_MLE = 75.109076
beta_MLE  = 1.466001


In [5]:
#1b. Grid Evaluation - Part 1: Calculate values and save them

# Use very few points to minimize memory usage
alpha_vals = np.linspace(alpha_mom*0.7, alpha_mom*1.3, 8)  # Only 8 points
beta_vals = np.linspace(beta_mom*0.7, beta_mom*1.3, 8)     # Only 8 points

# Calculate and store values one by one
print("Calculating values...")
Z = np.zeros((len(beta_vals), len(alpha_vals)))
calculated_values = []  # Store as list of (alpha, beta, value) tuples

for i, b in enumerate(beta_vals):
    for j, a in enumerate(alpha_vals):
        try:
            value = neg_log_likelihood((a, b), data)
            Z[i, j] = value
            calculated_values.append((a, b, value))
            print(f"Calculated point {i*len(alpha_vals)+j+1}/{len(beta_vals)*len(alpha_vals)}: ({a:.4f}, {b:.4f}) = {value:.4f}")
        except Exception as e:
            print(f"\nError at alpha={a:.4f}, beta={b:.4f}: {e}")
            Z[i, j] = np.nan

print("\nCalculation complete!")

Calculating values...
Calculated point 1/64: (52.4219, 1.0292) = 2115.1847
Calculated point 2/64: (58.8410, 1.0292) = 1693.5616
Calculated point 3/64: (65.2600, 1.0292) = 1342.7057
Calculated point 4/64: (71.6790, 1.0292) = 1055.5784
Calculated point 5/64: (78.0980, 1.0292) = 826.4164
Calculated point 6/64: (84.5170, 1.0292) = 650.4133
Calculated point 7/64: (90.9360, 1.0292) = 523.4996
Calculated point 8/64: (97.3550, 1.0292) = 442.1850
Calculated point 9/64: (52.4219, 1.1553) = 1553.6323
Calculated point 10/64: (58.8410, 1.1553) = 1206.1571
Calculated point 11/64: (65.2600, 1.1553) = 929.4491
Calculated point 12/64: (71.6790, 1.1553) = 716.4697
Calculated point 13/64: (78.0980, 1.1553) = 561.4555
Calculated point 14/64: (84.5170, 1.1553) = 459.6003
Calculated point 15/64: (90.9360, 1.1553) = 406.8345
Calculated point 16/64: (97.3550, 1.1553) = 399.6678
Calculated point 17/64: (52.4219, 1.2813) = 1158.9115
Calculated point 18/64: (58.8410, 1.2813) = 877.8992
Calculated point 19/64: (6

In [6]:
#1b. Grid Evaluation - Part 2: Ultra-simple visualization

# Extract the calculated values from the previous cell
alphas = [x[0] for x in calculated_values]
betas = [x[1] for x in calculated_values]
values = [x[2] for x in calculated_values]

# Find the minimum value and its location
min_value = min(values)
min_index = values.index(min_value)
min_alpha = alphas[min_index]
min_beta = betas[min_index]

# Print the results instead of plotting
print("Grid Evaluation Results:")
print(f"Minimum negative log-likelihood: {min_value:.6f}")
print(f"Found at alpha = {min_alpha:.6f}, beta = {min_beta:.6f}")
print(f"MOM estimates: alpha = {alpha_mom:.6f}, beta = {beta_mom:.6f}")

# Print a simple text-based visualization
print("\nSimple visualization of negative log-likelihood values:")
unique_alphas = sorted(set(alphas))
unique_betas = sorted(set(betas), reverse=True)  # Reverse to match visual grid layout

# Create a text-based grid
print("\n" + " " * 10, end="")
for a in unique_alphas:
    print(f"{a:10.4f}", end="")
print("\n" + "-" * (10 + 10 * len(unique_alphas)))

for b in unique_betas:
    print(f"{b:10.4f}|", end="")
    for a in unique_alphas:
        # Find the value for this alpha, beta pair
        try:
            idx = [i for i, (aa, bb) in enumerate(zip(alphas, betas)) if aa == a and bb == b][0]
            val = values[idx]
            # Use a simple representation for the value
            print(f"{val:10.4f}", end="")
        except:
            print(f"{'N/A':10s}", end="")
    print()

Grid Evaluation Results:
Minimum negative log-likelihood: 395.701413
Found at alpha = 78.097997, beta = 1.407306
MOM estimates: alpha = 74.888490, beta = 1.470319

Simple visualization of negative log-likelihood values:

             52.4219   58.8410   65.2600   71.6790   78.0980   84.5170   90.9360   97.3550
------------------------------------------------------------------------------------------
    1.9114|  422.6059  398.3449  444.8511  555.0859  723.2859  944.6450 1215.0934 1531.1408
    1.7854|  471.6793  403.6353  406.3586  472.8104  597.2275  774.8036 1001.4690 1273.7335
    1.6594|  556.3344  441.3011  397.0349  416.4974  493.9252  624.5119  804.1879 1029.4631
    1.5333|  687.6609  521.9248  426.9559  395.7156  422.4406  502.3245  631.2978  805.8702
    1.4073|  881.1371  660.3471  510.3244  424.0303  395.7014  420.5315  494.4510  613.9695
    1.2813| 1158.9115  877.8992  667.6540  521.1375  432.5863  397.1940  410.8911  470.1872
    1.1553| 1553.6323 1206.1571  929.4491  71